In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
df = spark.table("databricks_ete.silver.customers_silver")

In [0]:
df = df.dropDuplicates(subset=['customer_id'])
display(df.limit(10))

In [0]:
df = df.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))
display(df.limit(10))       


In [0]:
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimCustomerKey, customer_id, create_date, update_date
                        from databricks_ete.gold.DimCustomers''')
                        
else:
    df_old = spark.sql('''select 0 DimCustomersKey, 0 customer_id, 0 create_date, 0 update_date
                          from databricks_ete.silver.customers_silver where 1=0''')

In [0]:
df_old = df_old.withColumnRenamed('DimCustomerKey', 'oldDimCustomerKey')\
               .withColumnRenamed('customer_id','old_customer_id')\
               .withColumnRenamed('create_date','old_create_date')\
               .withColumnRenamed('update_date','old_update_date')


In [0]:
display(df_old)

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')

In [0]:
display(df_join)

In [0]:
df_new = df_join.filter(df_join['oldDimCustomerKey'].isNull())

In [0]:
df_old = df_join.filter(df_join['oldDimCustomerKey'].isNotNull())

In [0]:
df_old = df_old.drop('oldDimCustomerKey','old_customer_id','old_update_date')

df_old =  df_old.withColumnRenamed('old_create_date','create_date')
df_old = df_old.withColumn("create_date",to_timestamp(col('create_date')))

df_old = df_old.withColumn('update_date', current_timestamp())


In [0]:
display(df_old)

In [0]:
display(df_new)

In [0]:
df_new = df_new.drop('oldDimCustomerKey','old_customer_id','old_update_date','old_create_date')

df_new = df_new.withColumn('update_date', current_timestamp())
df_new = df_new.withColumn('create_date', current_timestamp())

In [0]:
display(df_new)

In [0]:
df_new = df_new.withColumn('DimCustomerKey', monotonically_increasing_id()+lit(1))

In [0]:
display(df_new)

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql('''select max(DimCustomerKey) as max_surrogate_key from databricks_ete.gold.DimCustomers''')
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']


In [0]:
df_new = df_new.withColumn('DimCustomerKey',lit(max_surrogate_key)+col('DimCustomerKey'))
display(df_new)

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
display(df_final)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_ete.gold.DimCustomers"):
    dlt_obj = DeltaTable.forPath(spark,"abfss://gold@databricksetestorage2003.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("trg").merge(df_final.alias("src"),"trg.DimCustomerKey = src.DimCustomerKey")\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()


else:
   
    df_final.write.mode("overwrite")\
    .option("path","abfss://gold@databricksetestorage2003.dfs.core.windows.net/DimCustomers")\
    .saveAsTable("databricks_ete.gold.DimCustomers")

In [0]:
%sql
select * from databricks_ete.gold.dimcustomers